In [1]:
%pwd

'c:\\Users\\timil\\OneDrive\\Documents\\My_ML project\\End-to-end-chicken-Disease-Classification\\research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\timil\\OneDrive\\Documents\\My_ML project\\End-to-end-chicken-Disease-Classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen= True)
class PrepareBasicModelConfig:
    root_dir : Path
    base_model_path : Path
    updated_base_model_path : Path
    params_image_size : list
    params_learning_rate : float
    params_include_top : bool
    params_weights : str
    params_classes : int


In [6]:
from src.Chicken_Disease_Classification.constant import *
from src.Chicken_Disease_Classification.utils.common import read_yaml, create_directories 

In [7]:

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([Path(self.config.artifacts_root)])

    def get_data_ingestion_config(self) -> PrepareBasicModelConfig:
        config = self.config.prepare_base_model

        # Create the directory where the ZIP will be downloaded and extracted
        create_directories([
            Path([config.root_dir])
        ])

        prepare_base_model_config = PrepareBasicModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path= Path(config.base_model_path),
            updated_base_model_path= Path(config.updated_base_model_path),
            params_image_size= self.params.IMAGE_SIZE,
            params_learning_rate= self.params.LEARNING_RATE,
            params_include_top= self.params.INCLUDE_TOP,
            params_weights= self.params.WEIGHTS,
            params_classes= self.params.CLASSES
            
        )
        return PrepareBasicModelConfig

In [ ]:
import os
import urllib.request as request 
from src.Chicken_Disease_Classification.utils.logger import logger
from src.Chicken_Disease_Classification.utils.common import get_size
import tensorflow as tf
from zipfile import ZipFile

In [15]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBasicModelConfig):
        self.config = config
    

    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape = self.config.params_image_size,
            weights = self.config.params_weights,
            include_top = self.config.params_include_top
        )
        self.save_model(Path= self.config.base_model_path, model= self.model)

@staticmethod
def _prepare_full_moodel(model, classes, freeze_all, freeze_till, learning_rate):
    if freeze_all:
        for layer in model.layers:
            model.trainable = False
    elif (freeze_all is not None ) and (freeze_till > 0):
        for layer in model.layers[ : -freeze_till]:
            model.trainable = False

    flatten_in = tf.keras.layer.flatten()(model.output)
    prediction = tf.keras.layers.Dense(
        units= classes,
        activation = "softmax"
    )(flatten_in)

    full_model = tf.keras.models.Model(
        inputs= model.input,
        outputs = prediction
    )

    full_model.compile(
        optimizer = tf.keras.optimizers.SGD(learning_rate= learning_rate),
        loss = tf.keras.losses.CategoricalCrossentropy(),
        metrics = ["accuracy"]
    
    )

    full_model.summary()
    return full_model
